In [1]:
pip install ultralytics

In [2]:
import cv2
import time
import numpy as np
from ultralytics import YOLO


INPUT_VIDEO    = "sample_video.mp4"
OUTPUT_VIDEO   = "yolo_output.mp4"
MODEL_NAME     = "yolov8m-seg.pt"
CONF_THRESHOLD = 0.5



print("Loading YOLOv8m-Seg model...")
model = YOLO(MODEL_NAME)
print("YOLOv8m-Seg loaded!\n")


cap = cv2.VideoCapture(INPUT_VIDEO)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in       = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (width, height))


all_confidences = []
fps_list        = []
frame_count     = 0

print("Running inference...\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    start_time = time.time()

    results = model.predict(source=frame, conf=CONF_THRESHOLD, verbose=False)
    result  = results[0]

    end_time  = time.time()
    fps_frame = 1.0 / (end_time - start_time)
    fps_list.append(fps_frame)

    boxes = result.boxes
    masks = result.masks
    names = model.names

    for i in range(len(boxes)):
        conf     = float(boxes.conf[i])
        cls_id   = int(boxes.cls[i])
        cls_name = names[cls_id]

        all_confidences.append(conf)

        np.random.seed(cls_id)
        colour = tuple(int(x) for x in np.random.randint(50, 230, size=3))

        if masks is not None:
            mask = cv2.resize(
                masks.data[i].cpu().numpy(),
                (width, height),
                interpolation=cv2.INTER_NEAREST
            )
            overlay = frame.copy()
            overlay[mask > 0] = colour
            frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

        x1, y1, x2, y2 = map(int, boxes.xyxy[i])
        cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)

        label = f"{cls_name} {conf:.2f}"
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2, cv2.LINE_AA)

    cv2.putText(frame, f"FPS: {fps_frame:.1f}", (12, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

    writer.write(frame)

    if frame_count % 30 == 0:
        print(f"  Frame {frame_count} / {total_frames}  |  FPS: {fps_frame:.1f}")

cap.release()
writer.release()


print("\n--------------------------------------")
print("  YOLOv8m-Seg — RESULTS SUMMARY")
print("--------------------------------------")
print(f"  Frames processed : {frame_count}")
print(f"  Average FPS      : {np.mean(fps_list):.2f}")
print(f"  Min FPS          : {np.min(fps_list):.2f}")
print(f"  Max FPS          : {np.max(fps_list):.2f}")

if all_confidences:
    print(f"\n  Total detections : {len(all_confidences)}")
    print(f"  Avg confidence   : {np.mean(all_confidences):.3f}")
    print(f"  Min confidence   : {np.min(all_confidences):.3f}")
    print(f"  Max confidence   : {np.max(all_confidences):.3f}")

print(f"\n  Output saved to  : {OUTPUT_VIDEO}")
print("--------------------------------------")

Loading YOLOv8m-Seg model...
YOLOv8m-Seg loaded!

Running inference...

  Frame 30 / 361  |  FPS: 27.5
  Frame 60 / 361  |  FPS: 23.9
  Frame 90 / 361  |  FPS: 27.4
  Frame 120 / 361  |  FPS: 27.3
  Frame 150 / 361  |  FPS: 27.5
  Frame 180 / 361  |  FPS: 24.5
  Frame 210 / 361  |  FPS: 27.1
  Frame 240 / 361  |  FPS: 27.6
  Frame 270 / 361  |  FPS: 27.2
  Frame 300 / 361  |  FPS: 27.4
  Frame 330 / 361  |  FPS: 27.5
  Frame 360 / 361  |  FPS: 27.2

--------------------------------------
  YOLOv8m-Seg — RESULTS SUMMARY
--------------------------------------
  Frames processed : 361
  Average FPS      : 26.74
  Min FPS          : 0.45
  Max FPS          : 28.46

  Total detections : 1658
  Avg confidence   : 0.865
  Min confidence   : 0.501
  Max confidence   : 0.971

  Output saved to  : yolo_output.mp4
--------------------------------------


In [3]:
pip install torch torchvision opencv-python

In [4]:
import cv2
import time
import numpy as np
import torch
import torchvision
from torchvision.transforms import functional as F

INPUT_VIDEO    = "sample_video.mp4"
OUTPUT_VIDEO   = "maskrcnn_output.mp4"
CONF_THRESHOLD = 0.5

# COCO class names
COCO_CLASSES = [
    "__background__", "person", "bicycle", "car", "motorcycle", "airplane",
    "bus", "train", "truck", "boat", "traffic light", "fire hydrant", "N/A",
    "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse",
    "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "N/A", "backpack",
    "umbrella", "N/A", "N/A", "handbag", "tie", "suitcase", "frisbee", "skis",
    "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
    "skateboard", "surfboard", "tennis racket", "bottle", "N/A", "wine glass",
    "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich",
    "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake",
    "chair", "couch", "potted plant", "bed", "N/A", "dining table", "N/A",
    "N/A", "toilet", "N/A", "tv", "laptop", "mouse", "remote", "keyboard",
    "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator",
    "N/A", "book", "clock", "vase", "scissors", "teddy bear", "hair drier",
    "toothbrush"
]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Loading Mask R-CNN model...")

model = torchvision.models.detection.maskrcnn_resnet50_fpn(
    weights=torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.DEFAULT
)
model.to(device)
model.eval()
print("Model loaded!\n")

cap = cv2.VideoCapture(INPUT_VIDEO)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in       = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (width, height))

all_confidences = []
fps_list        = []
frame_count     = 0

print("Running inference...\n")

with torch.no_grad():
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        start_time = time.time()

        rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        tensor = F.to_tensor(rgb).to(device)

        predictions = model([tensor])
        pred        = predictions[0]

        end_time  = time.time()
        fps_frame = 1.0 / (end_time - start_time)
        fps_list.append(fps_frame)


        boxes  = pred["boxes"].cpu().numpy()
        scores = pred["scores"].cpu().numpy()
        labels = pred["labels"].cpu().numpy()
        masks  = pred["masks"].cpu().numpy()

        for i in range(len(scores)):
            conf     = float(scores[i])
            cls_id   = int(labels[i])
            cls_name = COCO_CLASSES[cls_id] if cls_id < len(COCO_CLASSES) else "unknown"

            if conf < CONF_THRESHOLD or cls_name in ("N/A", "__background__"):
                continue

            all_confidences.append(conf)

            np.random.seed(cls_id)
            colour = tuple(int(x) for x in np.random.randint(50, 230, size=3))

            binary_mask = masks[i, 0] > 0.5
            overlay = frame.copy()
            overlay[binary_mask] = colour
            frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

            x1, y1, x2, y2 = map(int, boxes[i])
            cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)

            label = f"{cls_name} {conf:.2f}"
            cv2.putText(frame, label, (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2, cv2.LINE_AA)


        cv2.putText(frame, f"FPS: {fps_frame:.1f}", (12, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

        writer.write(frame)

        if frame_count % 30 == 0:
            print(f"  Frame {frame_count} / {total_frames}  |  FPS: {fps_frame:.1f}")

cap.release()
writer.release()


print("\n--------------------------------------")
print("  Mask R-CNN — RESULTS SUMMARY")
print("--------------------------------------")
print(f"  Frames processed : {frame_count}")
print(f"  Average FPS      : {np.mean(fps_list):.2f}")
print(f"  Min FPS          : {np.min(fps_list):.2f}")
print(f"  Max FPS          : {np.max(fps_list):.2f}")

if all_confidences:
    print(f"\n  Total detections : {len(all_confidences)}")
    print(f"  Avg confidence   : {np.mean(all_confidences):.3f}")
    print(f"  Min confidence   : {np.min(all_confidences):.3f}")
    print(f"  Max confidence   : {np.max(all_confidences):.3f}")

print(f"\n  Output saved to  : {OUTPUT_VIDEO}")
print("--------------------------------------")

Using device: cuda
Loading Mask R-CNN model...
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:01<00:00, 136MB/s]


Model loaded!

Running inference...

  Frame 30 / 361  |  FPS: 5.5
  Frame 60 / 361  |  FPS: 5.4
  Frame 90 / 361  |  FPS: 5.2
  Frame 120 / 361  |  FPS: 4.8
  Frame 150 / 361  |  FPS: 5.3
  Frame 180 / 361  |  FPS: 5.2
  Frame 210 / 361  |  FPS: 5.5
  Frame 240 / 361  |  FPS: 5.5
  Frame 270 / 361  |  FPS: 5.3
  Frame 300 / 361  |  FPS: 5.5
  Frame 330 / 361  |  FPS: 5.5
  Frame 360 / 361  |  FPS: 5.5

--------------------------------------
  Mask R-CNN — RESULTS SUMMARY
--------------------------------------
  Frames processed : 361
  Average FPS      : 5.31
  Min FPS          : 1.26
  Max FPS          : 6.15

  Total detections : 2455
  Avg confidence   : 0.892
  Min confidence   : 0.500
  Max confidence   : 0.998

  Output saved to  : maskrcnn_output.mp4
--------------------------------------
